# Module 2B — Universal Object Construction

Takes the raw activations from Module 2A and applies two configurable thresholds to produce universal objects (pair masks + universal AST/Builtin masks).

## Parameters

Two knobs control what counts as a "real" neuron in a circuit:

1. **EPSILON** (signal strength) — minimum mean |activation| for a neuron to count as active. Higher = only strongly-firing neurons survive.
2. **CONSISTENCY** (prompt share) — minimum fraction of prompts where the neuron fired. Higher = only neurons that fire reliably across prompt variations survive.

## Input / Output

- **Input:** `activations_{name}.h5` from Module 2A
- **Output:** `universal_{name}.h5` — pair masks, universal AST/Builtin masks, Jaccard matrices

In [1]:
# Cell 1 – Setup
import subprocess, sys, os, shutil
for pkg in ["h5py", "numpy", "pandas", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── Detect environment ───────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import numpy as np, pandas as pd, h5py
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

Environment: Local


## Parameters

Change these two values to control how strict the circuit construction is.

| Parameter | Low value | High value |
|-----------|-----------|------------|
| **EPSILON** | Large circuits, includes weak signals | Small circuits, only strong activations |
| **CONSISTENCY** | Tolerates neurons that skip some prompts | Only neurons that fire in (almost) every prompt |

In [2]:
# Cell 2 – Parameters (EDIT THESE)
# =====================================================================
# EPSILON — Signal strength threshold
#   Mean |activation| must exceed this value for a neuron to count.
#   Examples: 0.001 (very permissive), 0.01, 0.1 (moderate), 0.5 (strict)
EPSILON = 0.05

# CONSISTENCY — Prompt share threshold
#   Fraction of prompts where the neuron must fire (|act| > 0).
#   Examples: 0.5 (half), 0.8 (default), 0.95, 1.0 (every single prompt)
CONSISTENCY = 0.80
# =====================================================================

# ── Input / Output ───────────────────────────────────────────────────────
ACTIVATIONS_FILE = "activations_115x100.h5"

if IN_COLAB:
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

INPUT_HDF5  = f"{DATA_DIR}/{ACTIVATIONS_FILE}"
INPUT_NAME  = ACTIVATIONS_FILE.replace("activations_", "").replace(".h5", "")
OUTPUT_HDF5 = f"{DATA_DIR}/universal_{INPUT_NAME}.h5"

print(f"EPSILON     : {EPSILON}")
print(f"CONSISTENCY : {CONSISTENCY}")
print(f"Input       : {INPUT_HDF5}")
print(f"Output      : {OUTPUT_HDF5}")

EPSILON     : 0.05
CONSISTENCY : 0.8
Input       : /Users/piotrwilam/Data/CSP-Atlas/activations_115x100.h5
Output      : /Users/piotrwilam/Data/CSP-Atlas/universal_115x100.h5


In [3]:
# Cell 3 – Load raw activations
from module2.io_utils import load_activations_hdf5

data = load_activations_hdf5(INPUT_HDF5)
pair_activations = data["pair_activations"]
metadata = data["metadata"]

print(f"Loaded {len(pair_activations)} pairs")
print(f"Metadata: {metadata}")

Loaded 1370 pairs
Metadata: {'input_file': 'prompts_115x100.parquet', 'model_id': 'openai/circuit-sparsity', 'n_layers': np.int64(8), 'n_pairs': np.int64(1370), 'ast_nodes': ['AnnAssign', 'Assert', 'Assign', 'AsyncFor', 'AsyncFunctionDef', 'AsyncWith', 'Attribute', 'AugAssign', 'BinOp', 'BoolOp', 'Break', 'Call', 'ClassDef', 'Compare', 'Continue', 'Delete', 'Dict', 'DictComp', 'ExceptHandler', 'For', 'FunctionDef', 'GeneratorExp', 'Global', 'If', 'IfExp', 'Import', 'ImportFrom', 'Lambda', 'ListComp', 'Nonlocal', 'Pass', 'Raise', 'Return', 'Set', 'SetComp', 'Slice', 'Starred', 'Subscript', 'Try', 'UnaryOp', 'While', 'With', 'Yield', 'YieldFrom'], 'builtin_objs': ['ArithmeticError', 'AttributeError', 'Exception', 'FileNotFoundError', 'ImportError', 'IndexError', 'KeyError', 'LookupError', 'MemoryError', 'NameError', 'NotImplementedError', 'OSError', 'OverflowError', 'RecursionError', 'RuntimeError', 'StopIteration', 'TypeError', 'ValueError', 'ZeroDivisionError', '_isolated_', 'abs', 'al

In [4]:
# Cell 4 – Apply thresholds → pair masks
import numpy as np

pair_masks = {}
pair_stats = []

for (ast_n, blt_o), raw in pair_activations.items():
    n_prompts = raw["n_prompts"]
    masks = {}
    for lid, layer_data in raw["layers"].items():
        mean_act = layer_data["activation_sum"] / n_prompts
        firing_rate = layer_data["firing_count"].astype(np.float32) / n_prompts

        # Both conditions must be met:
        #   1. Mean activation strength > EPSILON
        #   2. Fraction of prompts where neuron fired >= CONSISTENCY
        mask = (mean_act > EPSILON) & (firing_rate >= CONSISTENCY)
        masks[lid] = mask

        pair_stats.append({
            "ast_node": ast_n, "builtin_obj": blt_o,
            "layer": lid, "circuit_size": int(mask.sum()),
            "n_prompts": n_prompts,
        })
    pair_masks[(ast_n, blt_o)] = masks

stats_df = pd.DataFrame(pair_stats)
sizes = stats_df["circuit_size"].values

print(f"Pair masks built: {len(pair_masks)}")
print(f"Circuit sizes: mean={sizes.mean():.1f}, min={sizes.min()}, max={sizes.max()}")
print(f"Empty masks: {(sizes == 0).sum()} / {len(sizes)}")

Pair masks built: 1370
Circuit sizes: mean=141.7, min=48, max=334
Empty masks: 0 / 10960


In [5]:
# Cell 5 – Compute universal modules + Jaccard matrices
from module2.marginalization import UniversalModuleComputer
from module2.metrics import compute_jaccard_matrix

ast_nodes = sorted(set(a for a, _ in pair_masks))
builtin_objs = sorted(set(b for _, b in pair_masks))

computer = UniversalModuleComputer()
universal_masks = computer.compute_all(pair_masks, ast_nodes, builtin_objs)

# Jaccard matrices at representative layer
n_layers = max(max(lm.keys()) for lm in pair_masks.values()) + 1
rep_layer = min(4, n_layers - 1)
metrics = {}

if universal_masks.get("ast"):
    metrics["jaccard_ast_matrix"] = compute_jaccard_matrix(
        universal_masks["ast"], rep_layer)
    metrics["ast_names"] = sorted(universal_masks["ast"].keys())

if universal_masks.get("builtin"):
    metrics["jaccard_builtin_matrix"] = compute_jaccard_matrix(
        universal_masks["builtin"], rep_layer)
    metrics["builtin_names"] = sorted(universal_masks["builtin"].keys())

print(f"Universal AST modules    : {len(universal_masks['ast'])}")
print(f"Universal Builtin modules: {len(universal_masks['builtin'])}")

Universal AST modules    : 44
Universal Builtin modules: 71


In [6]:
# Cell 6 – Save universal objects
from module2.io_utils import save_atlas_hdf5

out_metadata = {
    "model_id"           : metadata.get("model_id", "openai/circuit-sparsity"),
    "n_layers"           : n_layers,
    "epsilon"            : EPSILON,
    "consistency_thresh" : CONSISTENCY,
    "n_pairs"            : len(pair_masks),
    "input_file"         : os.path.basename(INPUT_HDF5),
    "ast_nodes"          : ast_nodes,
    "builtin_objs"       : builtin_objs,
}

save_atlas_hdf5(OUTPUT_HDF5, pair_masks, universal_masks, metrics, out_metadata)
print(f"Saved: {OUTPUT_HDF5}")

# Verification
with h5py.File(OUTPUT_HDF5, "r") as f:
    def _walk(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: {obj.shape} {obj.dtype}")
    f.visititems(_walk)
    print(f"\nMetadata: {dict(f['metadata'].attrs)}")

Saved: /Users/piotrwilam/Data/CSP-Atlas/universal_115x100.h5
  metadata/ast_nodes: (44,) object
  metadata/builtin_objs: (71,) object
  metrics/jaccard_ast_matrix: (44, 44) float64
  metrics/jaccard_builtin_matrix: (71, 71) float64
  pair_masks/layer_0/AnnAssign__abs: (2048,) bool
  pair_masks/layer_0/AnnAssign__all: (2048,) bool
  pair_masks/layer_0/AnnAssign__any: (2048,) bool
  pair_masks/layer_0/AnnAssign__bool: (2048,) bool
  pair_masks/layer_0/AnnAssign__bytearray: (2048,) bool
  pair_masks/layer_0/AnnAssign__bytes: (2048,) bool
  pair_masks/layer_0/AnnAssign__callable: (2048,) bool
  pair_masks/layer_0/AnnAssign__classmethod: (2048,) bool
  pair_masks/layer_0/AnnAssign__complex: (2048,) bool
  pair_masks/layer_0/AnnAssign__dict: (2048,) bool
  pair_masks/layer_0/AnnAssign__dir: (2048,) bool
  pair_masks/layer_0/AnnAssign__filter: (2048,) bool
  pair_masks/layer_0/AnnAssign__float: (2048,) bool
  pair_masks/layer_0/AnnAssign__frozenset: (2048,) bool
  pair_masks/layer_0/AnnAssign

In [7]:
# Cell 7 – Summary
import json

report = {
    "epsilon"             : EPSILON,
    "consistency"         : CONSISTENCY,
    "n_pairs"             : len(pair_masks),
    "n_universal_ast"     : len(universal_masks["ast"]),
    "n_universal_builtin" : len(universal_masks["builtin"]),
    "mean_circuit_size"   : float(sizes.mean()),
    "output_file"         : os.path.basename(OUTPUT_HDF5),
}
print(json.dumps(report, indent=2))
print(f"\nModule 2B complete.")
print(f"To evaluate these universal objects, point module4 or module2_evaluation")
print(f"at: {OUTPUT_HDF5}")

{
  "epsilon": 0.05,
  "consistency": 0.8,
  "n_pairs": 1370,
  "n_universal_ast": 44,
  "n_universal_builtin": 71,
  "mean_circuit_size": 141.71195255474453,
  "output_file": "universal_115x100.h5"
}

Module 2B complete.
To evaluate these universal objects, point module4 or module2_evaluation
at: /Users/piotrwilam/Data/CSP-Atlas/universal_115x100.h5
